# Bitcoin Market Sentiment and Trader Performance Analysis
This notebook explores the relationship between trader performance (from Hyperliquid historical data) and Bitcoin market sentiment (from the Fear and Greed Index).

## Phase 3: Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

## Phase 4: Load Data

In [ ]:
trades = pd.read_csv('../data/historical_data.csv')
sentiment = pd.read_csv('../data/fear_greed_index.csv')

## Phase 5: Understand the Data

In [ ]:
print("--- Trades Head ---")
display(trades.head())
print("\n--- Trades Info ---")
trades.info()
print("\n--- Trades Describe ---")
display(trades.describe())

In [ ]:
print("--- Sentiment Head ---")
display(sentiment.head())
print("\n--- Sentiment Info ---")
sentiment.info()
print("\n--- Sentiment Describe ---")
display(sentiment.describe())

## Phase 6: Check Missing Values

In [ ]:
print("Trades Missing Values:\n", trades.isnull().sum())
print("\nSentiment Missing Values:\n", sentiment.isnull().sum())

## Phase 7: Check Duplicates

In [ ]:
print("Trades Duplicated Rows:", trades.duplicated().sum())
print("Sentiment Duplicated Rows:", sentiment.duplicated().sum())

## Phase 8: Convert Dates
Convert timestamps to daily date format to allow merging.

In [ ]:
# Convert trades timestamp (epoch milliseconds) to date
trades['date'] = pd.to_datetime(trades['Timestamp'], unit='ms').dt.date

# Convert sentiment date string to date
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

## Phase 9: Merge Datasets

In [ ]:
merged_df = pd.merge(trades, sentiment, on='date', how='left')
print("Merged dataset shape:", merged_df.shape)
merged_df.head()

## Phase 10: Exploratory Data Analysis (EDA)

In [ ]:
# Analysis 1: How many trades happened?
num_trades = merged_df.shape[0]
print(f"Analysis 1 - Total Trades: {num_trades}")

# Analysis 2: How many traders?
num_traders = merged_df['Account'].nunique()
print(f"Analysis 2 - Total Unique Traders: {num_traders}")

# Analysis 3: How many unique coins?
num_coins = merged_df['Coin'].nunique()
print(f"Analysis 3 - Unique Coins: {num_coins}")

# Analysis 4: Total Profit
total_profit = merged_df['Closed PnL'].sum()
print(f"Analysis 4 - Total Profit (USD): {total_profit:,.2f}")

# Analysis 5: Average Profit
avg_profit = merged_df['Closed PnL'].mean()
print(f"Analysis 5 - Average Profit (USD): {avg_profit:,.2f}")

In [ ]:
# Analysis 6: Profit by Sentiment
profit_by_sentiment = merged_df.groupby('classification')['Closed PnL'].sum()
print("Analysis 6 - Profit by Sentiment (USD):")
print(profit_by_sentiment.map('{:,.2f}'.format))

# Analysis 7: Average Profit by Sentiment
avg_profit_by_sentiment = merged_df.groupby('classification')['Closed PnL'].mean()
print("\nAnalysis 7 - Average Profit by Sentiment (USD):")
print(avg_profit_by_sentiment.map('{:,.2f}'.format))

# Analysis 8: Number of Trades by Sentiment
trades_by_sentiment = merged_df['classification'].value_counts()
print("\nAnalysis 8 - Number of Trades by Sentiment:")
print(trades_by_sentiment)

In [ ]:
# Analysis 9: Average Leverage
# Check if leverage exists in dataset columns
leverage_col = [col for col in merged_df.columns if 'leverage' in col.lower()]
if leverage_col:
    print(f"Analysis 9 - Average Leverage: {merged_df[leverage_col[0]].mean():.2f}x")
    print(f"Analysis 10 - Maximum Leverage: {merged_df[leverage_col[0]].max():.2f}x")
else:
    print("Analysis 9 & 10 - Leverage column not present in the dataset (skipping).")

In [ ]:
# Analysis 11: Buy vs Sell Performance
buy_sell_perf = merged_df.groupby('Side')['Closed PnL'].agg(['count', 'sum', 'mean'])
print("Analysis 11 - Buy vs Sell Performance:")
display(buy_sell_perf)

In [ ]:
# Analysis 12: Profit by Coin
profit_by_coin = merged_df.groupby('Coin')['Closed PnL'].sum().sort_values(ascending=False)
print("Analysis 12 - Top 10 Most Profitable Coins (USD):")
print(profit_by_coin.head(10).map('{:,.2f}'.format))
print("\nWorst 10 Least Profitable Coins (USD):")
print(profit_by_coin.tail(10).map('{:,.2f}'.format))

In [ ]:
# Analysis 13: Top 10 Traders
top_traders = merged_df.groupby('Account')['Closed PnL'].sum().sort_values(ascending=False)
print("Analysis 13 - Top 10 Traders by Profit (USD):")
print(top_traders.head(10).map('{:,.2f}'.format))

# Analysis 14: Worst 10 Traders
print("\nAnalysis 14 - Worst 10 Traders by Profit (USD):")
print(top_traders.tail(10).map('{:,.2f}'.format))

In [ ]:
# Analysis 15: Largest Winning Trade
idx_max = merged_df['Closed PnL'].idxmax()
print("Analysis 15 - Largest Winning Trade:")
display(merged_df.loc[idx_max][['Account', 'Coin', 'Side', 'Closed PnL', 'Size USD', 'Timestamp IST']])

# Analysis 16: Largest Losing Trade
idx_min = merged_df['Closed PnL'].idxmin()
print("\nAnalysis 16 - Largest Losing Trade:")
display(merged_df.loc[idx_min][['Account', 'Coin', 'Side', 'Closed PnL', 'Size USD', 'Timestamp IST']])

In [ ]:
# Analysis 17: Trading Volume by Sentiment
vol_by_sentiment = merged_df.groupby('classification')['Size USD'].sum()
print("Analysis 17 - Trading Volume by Sentiment (USD):")
print(vol_by_sentiment.map('{:,.2f}'.format))

# Analysis 18: Average Position Size
avg_pos_size = merged_df['Size USD'].mean()
print(f"\nAnalysis 18 - Average Position Size (USD): {avg_pos_size:,.2f}")

In [ ]:
# Analysis 19: Winning Percentage
win_trades = merged_df[merged_df['Closed PnL'] > 0]
win_rate = (len(win_trades) / len(merged_df)) * 100
print(f"Analysis 19 - Overall Winning Percentage: {win_rate:.2f}%")

# Win rate by sentiment
print("\nWin Rate by Sentiment:")
for name, group in merged_df.groupby('classification'):
    group_win_rate = (len(group[group['Closed PnL'] > 0]) / len(group)) * 100
    print(f"  {name}: {group_win_rate:.2f}%")

In [ ]:
# Analysis 20: Profit Distribution
print("Analysis 20 - Profit (Closed PnL) Distribution Summary:")
display(merged_df['Closed PnL'].describe())

## Phase 11: Visualization

In [ ]:
# Ensure outputs directory exists
os.makedirs('../outputs/plots', exist_ok=True)

In [ ]:
# Graph 1: Fear vs Greed Profit
plt.figure(figsize=(10,6))
sns.barplot(x=profit_by_sentiment.index, y=profit_by_sentiment.values, palette='RdYlGn')
plt.title('Total Profit by Sentiment (USD)')
plt.ylabel('Total Profit')
plt.xlabel('Market Sentiment')
plt.tight_layout()
plt.savefig('../outputs/plots/fear_vs_greed_profit.png')
plt.show()

# Graph 2: Trade Count
plt.figure(figsize=(10,6))
sns.countplot(data=merged_df, x='classification', order=['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed'], palette='viridis')
plt.title('Trade Count by Sentiment')
plt.ylabel('Number of Trades')
plt.xlabel('Market Sentiment')
plt.tight_layout()
plt.savefig('../outputs/plots/trade_count.png')
plt.show()

In [ ]:
# Graph 3: Buy vs Sell Average Profit
plt.figure(figsize=(8,5))
sns.barplot(data=merged_df, x='Side', y='Closed PnL', palette='Set2')
plt.title('Average Profit by Trade Side')
plt.ylabel('Average Closed PnL (USD)')
plt.tight_layout()
plt.savefig('../outputs/plots/buy_vs_sell.png')
plt.show()

# Graph 4: Top 10 Traders
plt.figure(figsize=(12,6))
sns.barplot(x=top_traders.head(10).values, y=top_traders.head(10).index, palette='Blues_r')
plt.title('Top 10 Traders by Profit (USD)')
plt.xlabel('Total Closed PnL (USD)')
plt.ylabel('Account')
plt.tight_layout()
plt.savefig('../outputs/plots/top_10_traders.png')
plt.show()

In [ ]:
# Graph 5: Profit Distribution (excluding extreme outliers for visibility)
pnl_clean = merged_df['Closed PnL']
q_low = pnl_clean.quantile(0.01)
q_hi  = pnl_clean.quantile(0.99)
filtered_pnl = pnl_clean[(pnl_clean > q_low) & (pnl_clean < q_hi)]

plt.figure(figsize=(10,6))
sns.histplot(filtered_pnl, bins=50, kde=True, color='purple')
plt.title('Profit Distribution (Middle 98% of trades)')
plt.xlabel('Closed PnL (USD)')
plt.tight_layout()
plt.savefig('../outputs/plots/profit_distribution.png')
plt.show()

In [ ]:
# Graph 6: Leverage Distribution (if leverage is present)
if leverage_col:
    plt.figure(figsize=(10,6))
    sns.histplot(merged_df[leverage_col[0]].dropna(), bins=30, kde=True, color='teal')
    plt.title('Leverage Distribution')
    plt.xlabel('Leverage')
    plt.tight_layout()
    plt.savefig('../outputs/plots/leverage_distribution.png')
    plt.show()
else:
    print("Graph 6 skipped - Leverage column not present.")

In [ ]:
# Graph 7: Correlation Heatmap
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10,8))
sns.heatmap(merged_df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../outputs/plots/correlation_heatmap.png')
plt.show()

In [ ]:
# Graph 8: Daily Profit Trend
daily_profit = merged_df.groupby('date')['Closed PnL'].sum()
plt.figure(figsize=(14,6))
daily_profit.plot(color='darkgreen')
plt.title('Daily Profit Trend')
plt.ylabel('Closed PnL (USD)')
plt.xlabel('Date')
plt.grid(True)
plt.tight_layout()
plt.savefig('../outputs/plots/daily_profit_trend.png')
plt.show()

In [ ]:
# Graph 9: Most Traded Coins
top_coins = merged_df['Coin'].value_counts().head(10)
plt.figure(figsize=(10,6))
sns.barplot(x=top_coins.values, y=top_coins.index, palette='rocket')
plt.title('Top 10 Most Traded Coins (Trade Count)')
plt.xlabel('Number of Trades')
plt.ylabel('Coin')
plt.tight_layout()
plt.savefig('../outputs/plots/most_traded_coins.png')
plt.show()

In [ ]:
# Graph 10: Average Profit per Coin
avg_profit_coin = merged_df.groupby('Coin')['Closed PnL'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,6))
sns.barplot(x=avg_profit_coin.values, y=avg_profit_coin.index, palette='crest')
plt.title('Top 10 Coins by Average Profit per Trade (USD)')
plt.xlabel('Average Closed PnL (USD)')
plt.ylabel('Coin')
plt.tight_layout()
plt.savefig('../outputs/plots/avg_profit_per_coin.png')
plt.show()

## Phase 12: Insights
1. **Volume and Sentiment Correlation**: Analyze if trading volume peaks during greed/fear.
2. **PnL Variance**: Fear periods tend to show more negative outlier trades as retail sells panic or gets liquidated.
3. **Top vs. Bottom performance**: Highlight general coin profit trends.

## Phase 13: Conclusion
Summary of main takeaways for trading strategies based on sentiment.